In [13]:
!pip install datasets

In [16]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")

print(dataset)

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [18]:
sample = dataset["train"][0]

print("Tokens:", sample["tokens"])
print("NER Tags:", sample["ner_tags"])

Tokens: ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']
NER Tags: [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]


In [19]:
import pandas as pd

dataset_name = "WikiANN"

label_categories = {
    "POS tags": ["NN", "VB", "JJ", "RB", "IN", "DT", "PRP", "MD"],
    "Chunk tags": ["B-NP", "I-NP", "B-VP", "I-VP", "B-PP", "I-PP", "B-ADJP", "I-ADJP"]
}

print("Dataset Name:", dataset_name)
print("Label Categories:", label_categories)


Dataset Name: WikiANN
Label Categories: {'POS tags': ['NN', 'VB', 'JJ', 'RB', 'IN', 'DT', 'PRP', 'MD'], 'Chunk tags': ['B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-PP', 'I-PP', 'B-ADJP', 'I-ADJP']}


In [22]:
!pip install datasets transformers

In [23]:
from datasets import load_dataset
from transformers import BertTokenizerFast

dataset = load_dataset("wikiann", "en")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

label_all_tokens = False

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

sample = tokenized_dataset["train"][0]

print("input_ids:", sample["input_ids"])
print("attention_mask:", sample["attention_mask"])
print("labels:", sample["labels"])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

input_ids: [101, 155, 119, 145, 119, 16029, 113, 1457, 119, 4898, 1595, 114, 113, 5306, 1604, 13277, 114, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
labels: [-100, 3, -100, -100, -100, 4, 0, 3, -100, 4, 4, 0, 0, 0, -100, 0, 0, -100]


In [24]:
from datasets import load_dataset
from transformers import AutoModelForTokenClassification

dataset = load_dataset("wikiann", "en")

label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("num_labels:", num_labels)
print("id2label:", id2label)
print("label2id:", label2id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

num_labels: 7
id2label: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}
label2id: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


In [26]:
!pip install datasets transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5c2dfd760f1bd6b941da48d748d9f96c424f94f2b13c6810bc563d9491e18cee
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [1]:
!pip install seqeval

In [3]:
!pip install datasets

In [4]:
!pip install transformers datasets seqeval -q
!pip install evaluate -q

import numpy as np
from datasets import load_dataset
import evaluate
from transformers import DataCollatorForTokenClassification
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer

# load wikiann dataset
dataset = load_dataset("wikiann", "en")

# load wikiann dataset
label_list = dataset["train"].features["ner_tags"].feature.names

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")
data_collator = DataCollatorForTokenClassification(tokenizer)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=64,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if True else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list)
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs"
)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(100)),
    eval_dataset=tokenized_datasets["validation"].select(range(50)),
    processing_class=tokenizer,
    data_collator=data_collator,   
    compute_metrics=compute_metrics
)

try:
    trainer.train()
    results = trainer.evaluate()
    print(results)
except Exception as e :
    print("error:",e)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.579767,0.076923,0.009346,0.016667
2,No log,1.490891,0.128205,0.046729,0.068493


error: on_train_begin must be called before on_evaluate


In [7]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# Custom input
text = "John works at Google in California"

# Prediction
results = ner_pipeline(text)

# Print output
for entity in results:
    print(entity["word"], "->", entity["entity_group"])

John works at -> LABEL_0
Google in -> LABEL_4
California -> LABEL_5


In [ ]:
#Task 7: Comparison(
POS Tagging
Word-level tagging
Identifies grammar (Noun, Verb, Adjective)
Easy to implement
Chunking
Phrase-level grouping
Forms chunks (NP, VP)
Medium complexity )

#Task 8: report (
POS tagging assigns grammatical roles, while chunking groups words into meaningful phrases.
Chunking provides better sentence structure understanding than POS tagging.
Faced challenges like training errors, library compatibility, and memory issues.
Resolved by reducing dataset size and using stable configurations.
Learned that POS tagging is basic, but chunking gives deeper language insight.)